# Clean Leffingwell dataset (WITH CID)

In [1]:
import pyrfume
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

/opt/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


```
Info on leffingwell dataset in Pyrfume

[raw]
"leffingwell_data.csv" = "Odor labels for molecules as originally compiled by John Leffingwell and cleaned by Sanchez-Lengeling et al"
"leffingwell_readme.pdf" = "Information about the dataset"
LICENSE = "Licensing information and use restrictions according to the terms of John Leffingwell and Google"

[processed]
"molecules.csv" = "Information about odorant molecules used"
"behavior.csv" = "Odor labels for each molecule (one column per label)"
"behavior_sparse.csv" = "Odor labels for each molecule (single column)"
"stimuli.csv" = "Maps stimulus to CID, negative numbers assigned to compounds without CIDs"
```

In [2]:
# Load leffingwell datasets from pyrfume
molecules = pyrfume.load_data('leffingwell/molecules.csv', remote=True)
behavior = pyrfume.load_data('leffingwell/behavior.csv', remote=True)
behavior_sparse = pyrfume.load_data('leffingwell/behavior_sparse.csv', remote=True)

In [3]:
"""
Required descriptors based on the preprint:

Brian K. Lee, Emily J. Mayhew, Benjamin Sanchez-Lengeling,
Jennifer N. Wei, Wesley W. Qian, Kelsie Little, Matthew Andres,
Britney B. Nguyen, Theresa Moloy, Jane K. Parker, Richard C. Gerkin,
Joel D. Mainland, Alexander B. Wiltschko

`A Principal Odor Map Unifies Diverse Tasks in Human Olfactory Perception preprint
<https://www.biorxiv.org/content/10.1101/2022.09.01.504602v4>`_.
"""

required_desc = [
'alcoholic', 'aldehydic', 'alliaceous', 'almond', 'amber', 'animal',
'anisic', 'apple', 'apricot', 'aromatic', 'balsamic', 'banana', 'beefy',
'bergamot', 'berry', 'bitter', 'black currant', 'brandy', 'burnt',
'buttery', 'cabbage', 'camphoreous', 'caramellic', 'cedar', 'celery',
'chamomile', 'cheesy', 'cherry', 'chocolate', 'cinnamon', 'citrus', 'clean',
'clove', 'cocoa', 'coconut', 'coffee', 'cognac', 'cooked', 'cooling',
'cortex', 'coumarinic', 'creamy', 'cucumber', 'dairy', 'dry', 'earthy',
'ethereal', 'fatty', 'fermented', 'fishy', 'floral', 'fresh', 'fruit skin',
'fruity', 'garlic', 'gassy', 'geranium', 'grape', 'grapefruit', 'grassy',
'green', 'hawthorn', 'hay', 'hazelnut', 'herbal', 'honey', 'hyacinth',
'jasmin', 'juicy', 'ketonic', 'lactonic', 'lavender', 'leafy', 'leathery',
'lemon', 'lily', 'malty', 'meaty', 'medicinal', 'melon', 'metallic',
'milky', 'mint', 'muguet', 'mushroom', 'musk', 'musty', 'natural', 'nutty',
'odorless', 'oily', 'onion', 'orange', 'orangeflower', 'orris', 'ozone',
'peach', 'pear', 'phenolic', 'pine', 'pineapple', 'plum', 'popcorn',
'potato', 'powdery', 'pungent', 'radish', 'raspberry', 'ripe', 'roasted',
'rose', 'rummy', 'sandalwood', 'savory', 'sharp', 'smoky', 'soapy',
'solvent', 'sour', 'spicy', 'strawberry', 'sulfurous', 'sweaty', 'sweet',
'tea', 'terpenic', 'tobacco', 'tomato', 'tropical', 'vanilla', 'vegetable',
'vetiver', 'violet', 'warm', 'waxy', 'weedy', 'winey', 'woody'
]

### Analysis of molecules.csv

In [4]:
molecules.head()

,MolecularWeight,IsomericSMILES,IUPACName,name
CID,,,,
-955348933095,240.387,CCCCC=COC(=O)CCCCCCCC,NaN,Hexenyl nonanoate
-923209957509,196.290,CC(=O)OCC1C=CC(C(C)C)CC1,NaN,Tetrahydrocuminyl acetate
-874408321546,244.331,CCCCCCCCC(OC(C)=O)C(=O)OC,NaN,Methyl acetoxydecanoate
-873963935677,198.306,CCCCC=COC(=O)C(C)CCC,NaN,Hexenyl methylvalerate
-862841422647,148.271,CCCC(S)COCC,NaN,Ethoxymethylbutanethiol


In [5]:
molecules.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3522 entries, -955348933095 to 162353069
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   MolecularWeight  3522 non-null   float64
 1   IsomericSMILES   3522 non-null   object 
 2   IUPACName        3486 non-null   object 
 3   name             3522 non-null   object 
dtypes: float64(1), object(3)
memory usage: 137.6+ KB


In [6]:
# IMPORTANT: Save CID from index for later use
# CID is the index of molecules dataframe
cid_to_smiles = molecules['IsomericSMILES'].to_dict()
print(f"Total molecules with CID mapping: {len(cid_to_smiles)}")
print(f"Sample CID-SMILES mapping:")
for cid, smiles in list(cid_to_smiles.items())[:5]:
    print(f"  CID {cid}: {smiles}")

Total molecules with CID mapping: 3522
Sample CID-SMILES mapping:
  CID -955348933095: CCCCC=COC(=O)CCCCCCCC
  CID -923209957509: CC(=O)OCC1C=CC(C(C)C)CC1
  CID -874408321546: CCCCCCCCC(OC(C)=O)C(=O)OC
  CID -873963935677: CCCCC=COC(=O)C(C)CCC
  CID -862841422647: CCCC(S)COCC


### Analysis of behavior.csv

In [7]:
behavior.head()

,alcoholic,aldehydic,alliaceous,almond,animal,anisic,apple,apricot,aromatic,balsamic,...,tobacco,tomato,tropical,vanilla,vegetable,violet,warm,waxy,winey,woody
Stimulus,,,,,,,,,,,,,,,,,,,,,
-955348933095,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
-923209957509,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
-874408321546,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
-873963935677,0,0,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,0,0
-862841422647,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
behavior.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3522 entries, -955348933095 to 162353069
Columns: 113 entries, alcoholic to woody
dtypes: int64(113)
memory usage: 3.1 MB


### Analysis of behavior_sparse.csv

In [9]:
behavior_sparse.head()

,Raw Labels,Labels
Stimulus,,
-955348933095,"Herbal-green, waxy, oily, fruity","['green', 'oily', 'fruity', 'waxy', 'herbal']"
-923209957509,"Herbaceous, woody, slight spicy fruity odor","['woody', 'spicy', 'fruity', 'herbal']"
-874408321546,"delta-decalactone precursor; peach, apricot, b...","['peach', 'apricot', 'buttery']"
-873963935677,"Green, fruity, apple-like","['green', 'fruity', 'apple', 'tropical']"
-862841422647,Catty urine; cassis in dilution,['catty']


In [10]:
behavior_sparse.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3522 entries, -955348933095 to 162353069
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Raw Labels  3522 non-null   object
 1   Labels      3522 non-null   object
dtypes: object(2)
memory usage: 82.5+ KB


### Merge molecule structure with odor behavior labels

In [11]:
# merge molecules and behavior_sparse on CID (index)
mol_behaviors = molecules.merge(behavior_sparse, left_index=True, right_index=True, how='inner')

# Convert Labels column (which contains string representation of list) to semicolon-separated string
# The Labels column looks like: "['green', 'oily', 'fruity', 'waxy', 'herbal']"
mol_behaviors['Updated_Desc'] = mol_behaviors['Labels'].apply(lambda x: ';'.join(eval(x)))

# CRITICAL: Reset index to make CID a column instead of index
mol_behaviors = mol_behaviors.reset_index()
mol_behaviors = mol_behaviors.rename(columns={'index': 'CID'})

# Keep only the columns we need
mol_behaviors = mol_behaviors[['CID', 'IsomericSMILES', 'Updated_Desc']]
mol_behaviors.head()

,CID,IsomericSMILES,Updated_Desc
0,-955348933095,CCCCC=COC(=O)CCCCCCCC,green;oily;fruity;waxy;herbal
1,-923209957509,CC(=O)OCC1C=CC(C(C)C)CC1,woody;spicy;fruity;herbal
2,-874408321546,CCCCCCCCC(OC(C)=O)C(=O)OC,peach;apricot;buttery
3,-873963935677,CCCCC=COC(=O)C(C)CCC,green;fruity;apple;tropical
4,-862841422647,CCCC(S)COCC,catty


In [12]:
mol_behaviors.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3522 entries, 0 to 3521
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   CID             3522 non-null   int64 
 1   IsomericSMILES  3522 non-null   object
 2   Updated_Desc    3522 non-null   object
dtypes: int64(1), object(2)
memory usage: 82.7+ KB


### Split and clean descriptors

In [13]:
# (keep the same descriptor splitting logic as original)

In [14]:
def clean_descriptors(desc_str):
    """Clean and split descriptor string"""
    if pd.isna(desc_str):
        return []
    # Split by semicolon and strip whitespace
    descriptors = [d.strip().lower() for d in str(desc_str).split(';')]
    # Filter empty strings
    descriptors = [d for d in descriptors if d]
    return descriptors

# Apply cleaning
mol_behaviors['descriptors_list'] = mol_behaviors['Updated_Desc'].apply(clean_descriptors)

In [15]:
# Get all unique descriptors
all_descriptors = set()
for desc_list in mol_behaviors['descriptors_list']:
    all_descriptors.update(desc_list)

print(f"Total unique descriptors: {len(all_descriptors)}")
print(f"\nAll descriptors:")
print(sorted(all_descriptors))

Total unique descriptors: 113

All descriptors:
['alcoholic', 'aldehydic', 'alliaceous', 'almond', 'animal', 'anisic', 'apple', 'apricot', 'aromatic', 'balsamic', 'banana', 'beefy', 'berry', 'black currant', 'brandy', 'bread', 'brothy', 'burnt', 'buttery', 'cabbage', 'camphoreous', 'caramellic', 'catty', 'chamomile', 'cheesy', 'cherry', 'chicken', 'chocolate', 'cinnamon', 'citrus', 'cocoa', 'coconut', 'coffee', 'cognac', 'coumarinic', 'creamy', 'cucumber', 'dairy', 'dry', 'earthy', 'ethereal', 'fatty', 'fermented', 'fishy', 'floral', 'fresh', 'fruity', 'garlic', 'gasoline', 'grape', 'grapefruit', 'grassy', 'green', 'hay', 'hazelnut', 'herbal', 'honey', 'horseradish', 'jasmine', 'ketonic', 'leafy', 'leathery', 'lemon', 'malty', 'meaty', 'medicinal', 'melon', 'metallic', 'milky', 'mint', 'mushroom', 'musk', 'musty', 'nutty', 'odorless', 'oily', 'onion', 'orange', 'orris', 'peach', 'pear', 'phenolic', 'pine', 'pineapple', 'plum', 'popcorn', 'potato', 'pungent', 'radish', 'ripe', 'roasted'

In [16]:
# Filter to only required descriptors
print(f"Required descriptors: {len(required_desc)}")
print(f"Descriptors in dataset: {len(all_descriptors)}")
print(f"Overlap: {len(set(required_desc) & all_descriptors)}")

Required descriptors: 138
Descriptors in dataset: 113
Overlap: 105


### One-hot encode descriptors

In [17]:
# Create one-hot encoding for required descriptors
for desc in required_desc:
    mol_behaviors[desc] = mol_behaviors['descriptors_list'].apply(lambda x: 1 if desc in x else 0)

mol_behaviors.head()

/var/folders/cz/1hv_0jbx03v3cmxgyw9kd79w0000gn/T/ipykernel_3212/97041798.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mol_behaviors[desc] = mol_behaviors['descriptors_list'].apply(lambda x: 1 if desc in x else 0)
/var/folders/cz/1hv_0jbx03v3cmxgyw9kd79w0000gn/T/ipykernel_3212/97041798.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mol_behaviors[desc] = mol_behaviors['descriptors_list'].apply(lambda x: 1 if desc in x else 0)
/var/folders/cz/1hv_0jbx03v3cmxgyw9kd79w0000gn/T/ipykernel_3212/97041798.py:3: PerformanceWarn

,CID,IsomericSMILES,Updated_Desc,descriptors_list,alcoholic,aldehydic,alliaceous,almond,amber,animal,...,tropical,vanilla,vegetable,vetiver,violet,warm,waxy,weedy,winey,woody
0,-955348933095,CCCCC=COC(=O)CCCCCCCC,green;oily;fruity;waxy;herbal,"[green, oily, fruity, waxy, herbal]",0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,-923209957509,CC(=O)OCC1C=CC(C(C)C)CC1,woody;spicy;fruity;herbal,"[woody, spicy, fruity, herbal]",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,-874408321546,CCCCCCCCC(OC(C)=O)C(=O)OC,peach;apricot;buttery,"[peach, apricot, buttery]",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,-873963935677,CCCCC=COC(=O)C(C)CCC,green;fruity;apple;tropical,"[green, fruity, apple, tropical]",0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,-862841422647,CCCC(S)COCC,catty,[catty],0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
# Drop the temporary descriptors_list column
mol_behaviors_encoded = mol_behaviors.drop(columns=['descriptors_list'])
mol_behaviors_encoded.head()

,CID,IsomericSMILES,Updated_Desc,alcoholic,aldehydic,alliaceous,almond,amber,animal,anisic,...,tropical,vanilla,vegetable,vetiver,violet,warm,waxy,weedy,winey,woody
0,-955348933095,CCCCC=COC(=O)CCCCCCCC,green;oily;fruity;waxy;herbal,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,-923209957509,CC(=O)OCC1C=CC(C(C)C)CC1,woody;spicy;fruity;herbal,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,-874408321546,CCCCCCCCC(OC(C)=O)C(=O)OC,peach;apricot;buttery,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,-873963935677,CCCCC=COC(=O)C(C)CCC,green;fruity;apple;tropical,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,-862841422647,CCCC(S)COCC,catty,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
mol_behaviors_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3522 entries, 0 to 3521
Columns: 141 entries, CID to woody
dtypes: int64(139), object(2)
memory usage: 3.8+ MB


### Remove molecules with no required descriptors

In [20]:
# Count how many required descriptors each molecule has
descriptor_cols = [col for col in mol_behaviors_encoded.columns if col in required_desc]
mol_behaviors_encoded['desc_count'] = mol_behaviors_encoded[descriptor_cols].sum(axis=1)

print(f"Molecules with 0 descriptors: {(mol_behaviors_encoded['desc_count'] == 0).sum()}")
print(f"Molecules with ≥1 descriptors: {(mol_behaviors_encoded['desc_count'] > 0).sum()}")

Molecules with 0 descriptors: 23
Molecules with ≥1 descriptors: 3499


/var/folders/cz/1hv_0jbx03v3cmxgyw9kd79w0000gn/T/ipykernel_3212/1636998022.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  mol_behaviors_encoded['desc_count'] = mol_behaviors_encoded[descriptor_cols].sum(axis=1)


In [21]:
# Remove molecules with no descriptors
required_leffingwell_dataset = mol_behaviors_encoded[mol_behaviors_encoded['desc_count'] > 0].copy()
required_leffingwell_dataset = required_leffingwell_dataset.drop(columns=['desc_count'])
required_leffingwell_dataset = required_leffingwell_dataset.reset_index(drop=True)

print(f"Final dataset size: {len(required_leffingwell_dataset)}")
required_leffingwell_dataset.head()

Final dataset size: 3499


,CID,IsomericSMILES,Updated_Desc,alcoholic,aldehydic,alliaceous,almond,amber,animal,anisic,...,tropical,vanilla,vegetable,vetiver,violet,warm,waxy,weedy,winey,woody
0,-955348933095,CCCCC=COC(=O)CCCCCCCC,green;oily;fruity;waxy;herbal,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,-923209957509,CC(=O)OCC1C=CC(C(C)C)CC1,woody;spicy;fruity;herbal,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,-874408321546,CCCCCCCCC(OC(C)=O)C(=O)OC,peach;apricot;buttery,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,-873963935677,CCCCC=COC(=O)C(C)CCC,green;fruity;apple;tropical,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,-848964121442,CCCCCCCC=CC(=O)OC(CCCCCCCC)C(=O)O,milky,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [22]:
required_leffingwell_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3499 entries, 0 to 3498
Columns: 141 entries, CID to woody
dtypes: int64(139), object(2)
memory usage: 3.8+ MB


### Descriptor frequency analysis

In [23]:
# Frequency per descriptor in the required_leffingwell_dataset
# Exclude identifier columns
identifier_cols = ['CID', 'IsomericSMILES', 'Updated_Desc']
descriptor_cols = [col for col in required_leffingwell_dataset.columns if col not in identifier_cols]

descriptor_freq = required_leffingwell_dataset[descriptor_cols].sum().sort_values(ascending=False)
print("Descriptor frequencies:")
print(descriptor_freq)

Descriptor frequencies:
fruity    1391
green      907
sweet      825
floral     553
fatty      407
          ... 
rummy        0
juicy        0
jasmin       0
soapy        0
cortex       0
Length: 138, dtype: int64


### Save curated dataset with CID

In [24]:
# Reorder columns to put identifiers first (like goodscents)
identifier_cols = ['CID', 'IsomericSMILES', 'Updated_Desc']
descriptor_cols = [col for col in required_leffingwell_dataset.columns if col not in identifier_cols]
final_column_order = identifier_cols + descriptor_cols

required_leffingwell_dataset = required_leffingwell_dataset[final_column_order]

# Save the curated dataset with CID
required_leffingwell_dataset.to_csv('curated_leffingwell_with_cid.csv', index=False)

print("✅ Saved curated leffingwell dataset with CID!")
print(f"Dataset shape: {required_leffingwell_dataset.shape}")
print(f"First few columns: {required_leffingwell_dataset.columns.tolist()[:5]}")
print("\nFirst 3 rows:")
print(required_leffingwell_dataset[identifier_cols].head(3))

✅ Saved curated leffingwell dataset with CID!
Dataset shape: (3499, 141)
First few columns: ['CID', 'IsomericSMILES', 'Updated_Desc', 'alcoholic', 'aldehydic']

First 3 rows:
            CID             IsomericSMILES                   Updated_Desc
0 -955348933095      CCCCC=COC(=O)CCCCCCCC  green;oily;fruity;waxy;herbal
1 -923209957509   CC(=O)OCC1C=CC(C(C)C)CC1      woody;spicy;fruity;herbal
2 -874408321546  CCCCCCCCC(OC(C)=O)C(=O)OC          peach;apricot;buttery


### Verification: CID Statistics

In [25]:
print("\n" + "="*60)
print("VERIFICATION: CID STATISTICS")
print("="*60)

# Separate positive (real PubChem) and negative (placeholder) CIDs
positive_cids = required_leffingwell_dataset[required_leffingwell_dataset['CID'] > 0]
negative_cids = required_leffingwell_dataset[required_leffingwell_dataset['CID'] < 0]

print(f"\nTotal compounds: {len(required_leffingwell_dataset)}")
print(f"Compounds with real PubChem CID (positive): {len(positive_cids)}")
print(f"Compounds with placeholder CID (negative): {len(negative_cids)}")

if len(positive_cids) > 0:
    print(f"\nReal CID statistics:")
    print(positive_cids['CID'].describe())

print("\nSample of compounds with real CIDs:")
print(positive_cids[['CID', 'IsomericSMILES', 'Updated_Desc']].head(5))

print("\nSample of compounds with placeholder CIDs:")
print(negative_cids[['CID', 'IsomericSMILES', 'Updated_Desc']].head(5))


VERIFICATION: CID STATISTICS

Total compounds: 3499
Compounds with real PubChem CID (positive): 3465
Compounds with placeholder CID (negative): 34

Real CID statistics:
count    3.465000e+03
mean     6.974131e+06
std      2.197050e+07
min      4.000000e+00
25%      1.687200e+04
50%      9.098400e+04
75%      5.281553e+06
max      1.623531e+08
Name: CID, dtype: float64

Sample of compounds with real CIDs:
    CID         IsomericSMILES                       Updated_Desc
34    4                CC(CN)O                              fishy
35   58          CCC(=O)C(=O)O      creamy;caramellic;fatty;sweet
36   98        C(C(=O)C(=O)O)S             sulfurous;meaty;cheesy
37  101    C1=CC(=CC(=C1)O)C=O  phenolic;medicinal;aromatic;sweet
38  107  C1=CC=C(C=C1)CCC(=O)O                     balsamic;sweet

Sample of compounds with placeholder CIDs:
            CID                     IsomericSMILES  \
0 -955348933095              CCCCC=COC(=O)CCCCCCCC   
1 -923209957509           CC(=O)OCC1C=CC(C(